# 2 — Covalently modify a protein, watch it relax, and simulate it

The story in six moves:

1. **Load a protein** with full chemistry (notebook 1 wrote the input).
2. **Build a fragment from scratch** as star-sited SMILES. The `*` marks
   where the bond forms.
3. **Attach them.** One call places the fragment, removes one hydrogen per
   side, and forms the bond.
4. **Look at the result**, then **relax the fragment as a movie**. The
   protein stays fixed and only the fragment moves.
5. **Export** the modified PDB plus one bond record per new bond.
6. **Parameterize.** Those two outputs are all openff-pablo needs, and from
   there Interchange and OpenMM take over.

Run notebook 1 first, so `1ubq_protonated.pdb` exists.

In [1]:
from mbuild.biopolymers import Protein, prepare_fragment

import demo_utils

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues | net formal charge:",
      protein.net_formal_charge)

76 residues | net formal charge: 0


## 1. Prepare the fragment

Write the fragment as SMILES with a `*` at the atom that forms the bond.
`prepare_fragment` turns the star into a hydrogen, keeps the charges written
in the SMILES, gives every atom a PDB-style name, and records the bond site
in `link_atoms`. Here: an octanoyl group, starred at the carbonyl carbon.

In [2]:
fragment = prepare_fragment("*C(=O)CCCCCCC", "OCT")
print("bond site:", fragment.link_atoms)
print(fragment.n_particles, "atoms |",
      [particle.name for particle in fragment.particles()][:6], "...")

bond site: {'1': 'C1'}
25 atoms | ['H1', 'C1', 'O1', 'C2', 'C3', 'C4'] ...


## 2. Attach it

Name the protein site by residue number and atom name. The fragment already
knows its own bond site from the `*`. One hydrogen leaves each side, ports
along the two removed-hydrogen vectors align the fragment, and the bond
forms.

`attach` relaxes the fragment when it lands too close to the protein. Here
`relax=False` turns that off, because the next steps show the relaxation as
a movie. The warning about close atoms is the expected result of that
choice.

In [3]:
record = protein.attach(fragment, resnum=63, atom_name="NZ",
                        chain_id="A", relax=False)
print("new bond:", record.residue1.name, record.atom1_name, "-",
      record.residue2.name, record.atom2_name)
print("leaving hydrogens:", record.leaving1, record.leaving2)

2026-09-02 11:00:03,885 - mbuild.biopolymers.protein - WARNING - 1 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.43 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


new bond: LYS NZ - OCT C1
leaving hydrogens: ('HZ1',) ('H1',)


## 3. Look at the modification

`Protein.save_pdb` writes the file that every downstream loader reads, so
the view is built from that text. The lysine and the new OCT residue are
drawn in atom detail; the rest of the protein is a cartoon colored by
chain.

In [4]:
demo_utils.show_protein(protein, highlight_resnums=["63:A"],
                        highlight_resnames=["OCT"])

NGLWidget(layout=Layout(height='500px', width='700px'))

## 4. Relax the fragment as a movie

`protein.relax_fragments()` is the one-call form: it minimizes every HETATM
residue with mBuild's generic parameters while every other atom carries zero
mass, so the protein coordinates do not change.

`demo_utils.relax_movie` runs that same minimization in short bursts and
saves the coordinates after each burst, so the frames become a movie. Frame
0 is the rigid placement, before any minimization. Most of the motion
happens in the first frames, so the movie shows the fragment swing out of
the clash and settle.

Press play in the widget below.

In [5]:
import numpy as np

movie = demo_utils.relax_movie(protein, "octanoyl_relax.pdb",
                               n_frames=40, steps_per_frame=5)
path, frames, energies = movie
step = np.linalg.norm(frames[1:] - frames[:-1], axis=2).max(axis=1)
print(path, "|", len(frames), "frames |", frames.shape[1], "atoms")
print("largest atom step per frame (A), frames 1-3:", np.round(step[:3], 2))
print("potential energy (kJ/mol): first", round(float(energies[0])),
      "-> last", round(float(energies[-1])))

view = demo_utils.show_movie(movie, highlight_resnums=["63:A"],
                             highlight_resnames=["OCT"])
view

octanoyl_relax.pdb | 40 frames | 1254 atoms
largest atom step per frame (A), frames 1-3: [9.99 5.14 1.25]
potential energy (kJ/mol): first 285413 -> last 160669


NGLWidget(layout=Layout(height='500px', width='700px'), max_frame=39)

A GIF of the movie needs a live browser front end, because NGLView asks
the browser for each picture. `save_gif` writes a file only when the
environment variable `DEMO_RENDER` is set, so this cell does nothing in a
headless run.

In [6]:
demo_utils.save_gif(view, frames, "octanoyl_relax.gif")

## 5. Write the modified PDB and the bond records

`save_pdb` writes a standards-conformant file: residue names, real PDB
residue numbers, chain identifiers, HETATM for the fragment, TER after each
chain, and CONECT records only for the bonds that residue adjacency cannot
imply.

`bond_records()` returns one plain dict per new bond. It names the two
residues, the two bonded atoms, the hydrogens that left each side, and the
bond order. That is everything a downstream loader needs to know about the
modification.

In [7]:
protein.save_pdb("1ubq_octanoyl.pdb", overwrite=True)
records = protein.bond_records()
records

[{'residue_names': ('LYS', 'OCT'),
  'residue_numbers': (63, 77),
  'atom_names': ('NZ', 'C1'),
  'leaving_atoms': (['HZ1'], ['H1']),
  'bond_order': 1}]

## 6. Ingest with OpenFF Pablo

Pablo reads the PDB file against residue templates. It knows the CCD
residues, so it needs two things from us: a definition for OCT, and a
declaration of the crosslink.

`demo_utils.pablo_crosslink_kwargs` renames the keys of one bond record into
Pablo's `with_crosslink` vocabulary. No chemistry is added: the record
already holds it.

In [8]:
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb
from openff.toolkit import Molecule
from rdkit import Chem

# Build the OCT definition from the SAME starred SMILES, with the star
# replaced by hydrogen, exactly as prepare_fragment did. The atom order
# then matches, so the names transfer by position.
star = Chem.RWMol(Chem.MolFromSmiles("*C(=O)CCCCCCC"))
for atom in star.GetAtoms():
    if atom.GetAtomicNum() == 0:
        atom.SetAtomicNum(1)
mol = star.GetMol()
Chem.SanitizeMol(mol)
offmol = Molecule.from_rdkit(Chem.AddHs(mol), allow_undefined_stereo=True)

# Zip against the pristine `fragment`, not against the OCT residue inside
# the protein: attach() cloned the fragment and removed H1 from the clone,
# so the residue holds one atom less and every name after H1 would shift.
for atom, particle in zip(offmol.atoms, fragment.particles()):
    atom.name = particle.name

library = STD_CCD_CACHE.with_(
    {"OCT": [ResidueDefinition.from_molecule(offmol, residue_name="OCT")]}
).with_crosslink(**demo_utils.pablo_crosslink_kwargs(records[0]))

topology = topology_from_pdb("1ubq_octanoyl.pdb", residue_library=library)
molecule = topology.molecule(0)
print(molecule.n_atoms, "atoms | net charge:", molecule.total_charge)

1254 atoms | net charge: 0.0 elementary_charge


## 7. Assign force-field parameters

MosDef ships no biopolymer force field, so the OpenFF ecosystem takes over
here, using nothing but the objects above.

Charges come from the NAGL am1bcc graph model
(`openff-gnn-am1bcc-0.1.0-rc.3`) applied to the whole conjugate. That is one
call and it covers the non-standard residue, but it also replaces the ff14SB
library charges on every protein atom that ff14SB already describes.
Splitting the two — ff14SB on the protein, graph charges on the modified
site — is the next step for this notebook.

In [9]:
from openff.toolkit import ForceField
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

# TODO(wave 4): replace with demo_charges.assign_split_charges + build_interchange
conjugate = molecule  # the modified protein from Pablo, above
conjugate.assign_partial_charges("openff-gnn-am1bcc-0.1.0-rc.3.pt",
                                 toolkit_registry=NAGLToolkitWrapper())
print("NAGL am1bcc net charge:", conjugate.partial_charges.sum())

force_field = ForceField("ff14sb_off_impropers_0.0.4.offxml",
                         "openff-2.3.0.offxml")
print("force field handlers:", len(force_field.registered_parameter_handlers))

NAGL am1bcc net charge: 2.6645352591003757e-14 elementary_charge


force field handlers: 9


## 8. Solvate and run MD with OpenMM

The charges computed above are passed in as preset charges, so the force
field does not compute its own.

In [10]:
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.toolkit import Topology
from openff.units import unit as off_unit

water = Molecule.from_smiles("O")
water.generate_conformers(n_conformers=1)
for atom in water.atoms:
    atom.metadata["residue_name"] = "HOH"

solvated = pack_box([water], [1500],
                    solute=Topology.from_molecules([conjugate]),
                    target_density=0.95 * off_unit.gram / off_unit.milliliter,
                    box_shape=UNIT_CUBE,
                    tolerance=2.0 * off_unit.angstrom)
print("solvated:", solvated.n_atoms, "atoms")

interchange = force_field.create_interchange(
    solvated, charge_from_molecules=[conjugate])

solvated: 5754 atoms


In [11]:
import openmm
from openmm import unit

simulation = interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picosecond, 2.0 * unit.femtosecond),
)
simulation.minimizeEnergy(maxIterations=200)
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
for block in range(2):
    simulation.step(250)
    state = simulation.context.getState(getEnergy=True)
    print(f"step {(block + 1) * 250}: PE =", state.getPotentialEnergy())
print("MD ran: a covalently modified protein, built in mBuild, in OpenMM.")

step 250: PE = -67877.51931124476 kJ/mol


step 500: PE = -68073.24296484531 kJ/mol
MD ran: a covalently modified protein, built in mBuild, in OpenMM.


## Recap

- **mBuild** owned the coordinates: strict protein loading, fragment
  definition from SMILES, covalent attachment, fragment relaxation with the
  protein fixed, and chemistry-complete export.
- **OpenFF** owned the parameters: Pablo ingestion from the PDB file plus
  one bond record, NAGL am1bcc graph charges, ff14SB and Sage 2.3.0 through
  Interchange, then OpenMM.
- More demos — multi-site polymer tethering, glycans from GLYCAM PDBs,
  branched sugars, mixed force fields — live on this repository's
  `showcase-extras` branch.

---

## Appendix: a larger fragment

Nothing above is specific to a small fragment. This appendix repeats the
mBuild half with a sulfonated cyanine FRET dye, 106 atoms carrying one
positive and one negative formal charge, on a fresh copy of the protein. The dye lands with no clash, so it
needs no relaxation and no movie.

It stops at the charges. The parameterization from there is the same as
above.

In [12]:
DYE_SMILES = (
    "CC1(C2=C(C=CC(=C2)S(=O)(=O)O)[N+](=C1C=CC=CC=C3C(C4=C(N3CCCS(=O)(=O)[O-])"
    "C=CC(=C4)S(=O)(=O)O)(C)CCCCC(=O)NCC*)CCCS(=O)(=O)O)C"
)

dye_protein = Protein("1ubq_protonated.pdb")
dye = prepare_fragment(DYE_SMILES, "FRE")
print("dye:", dye.n_particles, "atoms | formal charge:", dye.formal_charge,
      "| bond site:", dye.link_atoms)

dye_record = dye_protein.attach(dye, resnum=63, atom_name="NZ", chain_id="A")
dye_protein.save_pdb("1ubq_FRET.pdb", overwrite=True)
dye_records = dye_protein.bond_records()
print("new bond:", dye_record.atom1_name, "-", dye_record.atom2_name,
      "| leaving:", dye_record.leaving1, dye_record.leaving2)

INFO:mbuild.biopolymers.fragments:Fragment 'Compound' wrapped into residue 'FRE'.


INFO:mbuild.biopolymers.fragments:Renamed atoms of residue FRE to element+index names so they are unique within the residue.


dye: 106 atoms | formal charge: 0 | bond site: {'1': 'C33'}
new bond: NZ - C33 | leaving: ('HZ1',) ('H1',)


`demo_utils.pablo_residue_library` performs the whole of step 6 in one
call: it builds the residue definition from the starred SMILES and adds one
crosslink declaration per bond record. It needs the pristine `dye`, for the
same reason the zip in step 6 did.

In [13]:
dye_library = demo_utils.pablo_residue_library(
    DYE_SMILES, dye, "FRE", dye_records)
dye_topology = topology_from_pdb("1ubq_FRET.pdb",
                                 residue_library=dye_library)
dye_conjugate = dye_topology.molecule(0)
print(dye_conjugate.n_atoms, "atoms | net charge:",
      dye_conjugate.total_charge)

dye_conjugate.assign_partial_charges("openff-gnn-am1bcc-0.1.0-rc.3.pt",
                                     toolkit_registry=NAGLToolkitWrapper())
print("NAGL am1bcc net charge:", dye_conjugate.partial_charges.sum())

1335 atoms | net charge: 0.0 elementary_charge


NAGL am1bcc net charge: -2.042810365310288e-14 elementary_charge


`show_protein` reads a PDB file as well as a `Protein`, so the same view
works on the file that was written.

In [14]:
demo_utils.show_protein("1ubq_FRET.pdb", highlight_resnames=["FRE"])

NGLWidget(layout=Layout(height='500px', width='700px'))